In [4]:
!pip install gdelt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.4/787.4 kB 41.7 MB/s eta 0:00:00


In [5]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
# import finnhub
import gdelt
import logging
from pandas_datareader import data as pdr
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

here


# 1. Create bronze_data (gdelt + crawl title)

In [ ]:
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime

def download_gkg_to_df(date_str):
    """
    Args: date_str (str): Format 'YYYYMMDD' (e.g., '20251219')
    """
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    logger.info(f"Downloading {url}")
    
    response = requests.get(url)
    if response.status_code == 200:
        # Open the zip file in memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            # GDELT zip files usually contain exactly one CSV file
            csv_filename = z.namelist()[0]
            with z.open(csv_filename) as f:
                # Use sep='\t' because GDELT GKG uses Tabs
                # Use header=None because raw GKG files don't have headers
                df = pd.read_csv(f, sep='\t', header=None, encoding='utf-8', on_bad_lines='skip')
                
                # Assigning standard GKG V2 columns (truncated for brevity)
                df.columns = ['GKGRECORDID', 'DATE', 'SourceCollectionIdentifier', 'SourceCommonName', 
                              'DocumentIdentifier', 'Counts', 'V2Counts', 'Themes', 'V2Themes', 
                              'Locations', 'V2Locations', 'Persons', 'V2Persons', 'Organizations', 
                              'V2Organizations', 'V2Tone', 'EnhancedDates', 'EnhancedLocations', 
                              'EnhancedPersons', 'EnhancedOrganizations', 'Unused', 'AllNames', 
                              'Amounts', 'TranslationInfo', 'ExtrasXML']
                return df
    else:
        logger.error(f"Failed to download. Status code: {response.status_code}")
        return None

# Usage in a cell:
# df_test = download_gkg_to_df('20251219')

In [2]:
from bs4 import BeautifulSoup
import time

def get_article_title(url, timeout=15, max_retries=3):
    """
    Fetch the title of an article from a URL with retry logic
    
    Args:
        url: The URL to fetch the title from
        timeout: Request timeout in seconds
        max_retries: Maximum number of retry attempts
    
    Returns:
        The article title or None if not found after all retries
    """
    for attempt in range(max_retries):
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            response = requests.get(url, timeout=timeout, headers=headers, allow_redirects=True)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Try different methods to find the title
            # 1. Look for <title> tag
            if soup.title and soup.title.string:
                return soup.title.string.strip()
            
            # 2. Look for og:title meta tag
            og_title = soup.find('meta', property='og:title')
            if og_title and og_title.get('content'):
                return og_title.get('content').strip()
            
            # 3. Look for twitter:title meta tag
            twitter_title = soup.find('meta', attrs={'name': 'twitter:title'})
            if twitter_title and twitter_title.get('content'):
                return twitter_title.get('content').strip()
            
            # 4. Look for h1 tag
            h1 = soup.find('h1')
            if h1:
                return h1.get_text().strip()
            
            return None
            
        except Exception as e:
            if attempt < max_retries - 1:
                logger.warning(f"Attempt {attempt + 1} failed for {url}: {str(e)}. Retrying...")
                time.sleep(2 * (attempt + 1))  # Exponential backoff
            else:
                logger.error(f"Failed to fetch title from {url} after {max_retries} attempts: {str(e)}")
                return None

# Add title column to dataframe
def crawl_titles(df, url_column='sourceurl', delay=2, timeout = 15, max_retries = 3):
    """
    Crawl titles for all URLs in the dataframe
    
    Args:
        df: The dataframe containing URLs
        url_column: Name of the column containing URLs
        delay: Delay between requests in seconds (to be polite)
    
    Returns:
        DataFrame with added 'title' column
    """
    titles = []
    
    for idx, row in df.iterrows():
        url = row[url_column]
        logger.info(f"Fetching title {idx+1}/{len(df)} from {url}")
        
        title = get_article_title(url=url, timeout=timeout, max_retries=max_retries)
        titles.append(title)
        
        # Add delay to avoid overwhelming servers
        if idx < len(df) - 1:
            time.sleep(delay)
    
    df['title'] = titles
    return df

In [3]:
def ingest_gdelt_events(days=1, max_records=None):
    """
    Ingest GDELT Events data from the last M days
    
    Args:
        days: Number of days to fetch data for (default: 1)
        max_records: Maximum number of records to fetch per day
    """
    try:
        logger.info(f"Starting GDELT Events ingestion for the last {days} days")
        
        gd = gdelt.gdelt(version=2)
        
        all_events = []
        
        # Fetch events for each day in the range
        for i in range(days):
            date = (datetime.now() - timedelta(days=i)).strftime('%Y %m %d')
            logger.info(f"Fetching GDELT events for {date}")
            
            try:
                results = gd.Search(date, table='events', coverage=True)
                
                if results is not None and not results.empty:
                    
                    if max_records is not None:
                        df = results.head(max_records)
                    else:
                        df = results

                    all_events.append(df)
                    logger.info(f"Fetched {len(df)} events for {date}")
                else:
                    logger.warning(f"No GDELT events found for {date}")
            except Exception as e:
                logger.error(f"Error fetching events for {date}: {str(e)}")
                continue
        
        if not all_events:
            logger.warning("No GDELT events found for the specified period")
            return
        
        
        combined_df = pd.concat(all_events, ignore_index=True)
        combined_df.columns = [col.lower() for col in combined_df.columns]
        
        # Crawl
        url_column = 'sourceurl'

        try:
            results_df = crawl_titles(combined_df, url_column=url_column, delay=2, timeout=15, max_retries=3)
            logger.info("Successfully crawled titles for GDELT events")
        except Exception as e:
            logger.error(f"Error crawling titles: {str(e)}")
            logger.info("Returning GDELT events without titles")
            return combined_df
        
        return results_df
        
    except Exception as e:
        logger.error(f"Error ingesting GDELT events: {str(e)}")
        raise


def ingest_gdelt_gkg(days=1, max_records=None):
    """
    Ingest GDELT GKG (Global Knowledge Graph) data from the last M days
    
    Args:
        days: Number of days to fetch data for (default: 1)
        max_records: Maximum number of records to fetch per day
    """
    try:
        logger.info(f"Starting GDELT GKG ingestion for the last {days} days")
        
        gd = gdelt.gdelt(version=2)
        
        all_gkg = []
        
        # Fetch GKG data for each day in the range
        for i in range(days):
            date = (datetime.now() - timedelta(days=i)).strftime('%Y %m %d')
            logger.info(f"Fetching GDELT GKG data for {date}")
            
            try:
                results = gd.Search(date, table='gkg', coverage=True)
                
                if results is not None and not results.empty:
                    # Limit records per day
                    if max_records is not None:
                        df = results.head(max_records)
                    else:
                        df = results

                    all_gkg.append(df)
                    logger.info(f"Fetched {len(df)} GKG records for {date}")
                else:
                    logger.warning(f"No GDELT GKG data found for {date}")
            except Exception as e:
                logger.error(f"Error fetching GKG data for {date}: {str(e)}")
                continue
        
        if not all_gkg:
            logger.warning("No GDELT GKG data found for the specified period")
            return
        
        
        combined_df = pd.concat(all_gkg, ignore_index=True)
        combined_df.columns = [col.lower() for col in combined_df.columns]
        
        # crawl
        url_column = 'documentidentifier'

        try:
            results_df = crawl_titles(combined_df, url_column=url_column, delay=2, timeout=15, max_retries=3)
            logger.info("Successfully crawled titles for GDELT events")
        except Exception as e:
            logger.error(f"Error crawling titles: {str(e)}")
            logger.info("Returning GDELT events without titles")
            return combined_df
        
        return results_df
        
        logger.info(f"Successfully ingested {len(combined_df)} GDELT GKG records from the last {days} days")
        
    except Exception as e:
        logger.error(f"Error ingesting GDELT GKG data: {str(e)}")
        raise

In [ ]:
def ingest_stooq(spark, symbols=['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'AMZN'], days=30):
    try:
        
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)
        
        all_data = []
        
        for symbol in symbols:
            try:
                logger.info(f"Fetching historical data for {symbol} from {start_date.date()} to {end_date.date()}")
                
                # Fetch historical data from Stooq
                df = pdr.DataReader(symbol, 'stooq', start_date, end_date)
                
                if not df.empty:
                    # Reset index to make Date a column
                    df = df.reset_index()
                    
                    # Add symbol column
                    df['symbol'] = symbol
                    
                    # Rename columns to lowercase and more descriptive names
                    df.columns = [col.lower() for col in df.columns]
                    df = df.rename(columns={'date': 'timestamp'})
                    
                    all_data.append(df)
                    logger.info(f"Fetched {len(df)} records for {symbol}")
                else:
                    logger.warning(f"No data available for {symbol}")
                        
            except Exception as e:
                logger.error(f"Error fetching data for {symbol}: {str(e)}")
                continue
        
        if all_data:
            combined_df = pd.concat(all_data, ignore_index=True)
    except:
        pass

    return combined_df

In [ ]:
# Option 1: Load from API (may timeout)
bronze_events = ingest_gdelt_events(days=2, max_records=50)
bronze_gkg = ingest_gdelt_gkg(days=2, max_records=50)
bronze_events.to_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_events.csv')
bronze_gkg.to_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_gkg.csv')


In [ ]:
bronze_events = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_events.csv', nrows=100)
bronze_events = crawl_titles(bronze_events, url_column='sourceurl', delay=1, timeout=15, max_retries=2)
logger.info(f"Loaded {len(bronze_events)} events from CSV with titles")

bronze_gkg = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_gkg.csv', nrows=100)
bronze_gkg = crawl_titles(bronze_gkg, url_column='documentidentifier', delay=1, timeout=15, max_retries=2)
logger.info(f"Loaded {len(bronze_gkg)} GKG records from CSV with titles")

bronze_stooq = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/stooq_stock_prices.csv', nrows=100)
logger.info(f"Loaded {len(bronze_stooq)} stock price records from CSV")

2025-12-20 11:13:25,062 - INFO - Fetching title 1/100 from https://cbs4local.com/newsletter-daily/socorro-isd-considers-2500-incentive-for-voluntary-employee-resignations
2025-12-20 11:13:27,170 - INFO - Fetching title 2/100 from https://www.govexec.com/management/2025/12/usda-received-overwhelmingly-negative-feedback-its-reorg-plan-employees-lawmakers-and-locals-governments/410143/
2025-12-20 11:13:29,959 - INFO - Fetching title 3/100 from https://economictimes.indiatimes.com/news/international/world-news/pakistans-ties-with-nordic-states-touch-new-low-as-norway-envoy-served-demarche/articleshow/125943113.cms
2025-12-20 11:13:30,069 - WARNING - Attempt 1 failed for https://economictimes.indiatimes.com/news/international/world-news/pakistans-ties-with-nordic-states-touch-new-low-as-norway-envoy-served-demarche/articleshow/125943113.cms: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')). Retrying...
2025-12-20 11:13:32,224 - ERROR - Failed to fetch title fro

In [6]:
nan_rows = bronze_events[bronze_events['title'].isna()]
print(len(nan_rows))
nan_rows[['sourceurl', 'title']]

8


,sourceurl,title
2,https://economictimes.indiatimes.com/news/inte...,None
7,https://economictimes.indiatimes.com/news/inte...,None
15,https://wcbm.com/national-headline/national-gu...,None
42,https://thegrio.com/2025/12/12/ebro-in-the-mor...,None
68,https://wcbm.com/national-headline/national-gu...,None
86,https://thegrio.com/2025/12/12/trump-attacks-r...,None
87,https://wcbm.com/national-headline/national-gu...,None
92,https://www.arkansasonline.com/news/2025/dec/1...,None


In [7]:
nan_rows = bronze_gkg[bronze_gkg['title'].isna()]
print(len(nan_rows))
nan_rows[['documentidentifier', 'title']]

31


,documentidentifier,title
2,https://www.wvik.org/illinois/2025-12-12/appea...,None
5,https://www.techdirt.com/2025/12/12/trump-pret...,None
7,https://www.southwhidbeyrecord.com/news/coupev...,None
9,"http://www.morningsun.net/stories/untitiled,28...",None
13,https://www.pcmag.com/news/whats-new-to-watch-...,None
14,https://www.techdirt.com/2025/12/12/how-cops-a...,None
16,https://www.fool.ca/2025/12/12/1-canadian-stoc...,None
19,https://www.pressreleasepoint.com/user/silos-a...,None
24,"http://www.morningsun.net/stories/untitiled,28...",None
27,https://radio.wpsu.org/2025-12-12/penn-state-s...,None


In [8]:
# Check columns and dtypes in bronze_data
print(bronze_events.shape)
print(bronze_gkg.shape)

(100, 63)
(100, 28)


In [9]:
bronze_events.to_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_events_with_titles.csv', index=False)
bronze_gkg.to_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_gkg_with_titles.csv', index=False)

In [10]:
bronze_events[['title']]

,title
0,"Socorro ISD considers $2,500 incentive for vol..."
1,USDA received overwhelmingly negative feedback...
2,None
3,Pakistan’s ties with Nordic states touch new l...
4,Pakistan’s ties with Nordic states touch new l...
...,...
95,Sydney man charged with threatening to kill co...
96,Sydney man charged with threatening to kill co...
97,Sydney man charged with threatening to kill co...
98,"76 Years Ago, This Film Noir Had Cinema's Best..."


In [11]:
bronze_gkg[['title']]

,title
0,Magic in the air as Lionel Messi reaches Kolkata
1,Falcon Gold (CVE:FG) Stock Price Down 25% – T...
2,None
3,How the Child Vaccine Schedule Could Change Un...
4,White House East Wing demolition: Preservation...
...,...
95,City of Midland to hold Merry Lights Parade th...
96,Army sergeant to face court-martial in Georgia...
97,Tate McRae Details ‘Overwhelming’ Response To ...
98,Passenger Arrested For Trying To Open Plane Do...


# 2. Create silver data: (preprocessing)

 - Drop rows where title is NaN.
 - Select relevant columns for financial prediction
 - Convert date to datetime
 - Sort by date

In [78]:
bronze_events = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_events_with_titles.csv', nrows=1000)
bronze_gkg = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_gkg_with_titles.csv', nrows=1000)
bronze_stooq = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/stooq_stock_prices.csv', nrows=1000)

bronze_events = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_events.csv', nrows=1000)
bronze_gkg = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/gdelt_gkg.csv', nrows=1000)
bronze_stooq = pd.read_csv('/mnt/c/Coding/bdsp-test/data-csv/stooq_stock_prices.csv', nrows=1000)

# create column title is ''
bronze_events['title'] = [''] * len(bronze_events)
bronze_gkg['title'] = [''] * len(bronze_gkg)

In [79]:
bronze_stooq.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume', 'symbol'], dtype='object')

In [80]:
# Process silver_events
logger.info("Processing silver_events...")

# Drop rows where title is NaN
silver_events = bronze_events.dropna(subset=['title']).copy()
logger.info(f"Dropped {len(bronze_events) - len(silver_events)} rows with missing titles")

# Select relevant columns for financial prediction
events_columns = [
    'globaleventid',
    'sqldate',
    'year',
    'monthyear',
    'eventcode',
    'cameocodedescription',
    'eventbasecode',
    'eventrootcode',
    'quadclass',
    'goldsteinscale',
    'nummentions',
    'numsources',
    'numarticles',
    'avgtone',
    'actor1name',
    'actor1countrycode',
    'actor2name',
    'actor2countrycode',
    'actiongeo_countrycode',
    'actiongeo_fullname',
    'actiongeo_lat',
    'actiongeo_long',
    'sourceurl',
    'title'
]

silver_events = silver_events[events_columns]

# Convert date to datetime
silver_events['date'] = pd.to_datetime(silver_events['sqldate'], format='%Y%m%d', errors='coerce')

# Drop rows with invalid dates
invalid_dates = silver_events['date'].isna().sum()
if invalid_dates > 0:
    logger.warning(f"Found {invalid_dates} invalid dates, dropping them")
    silver_events = silver_events.dropna(subset=['date'])

# Sort by date
silver_events = silver_events.sort_values('date').reset_index(drop=True)

# Remove duplicate events (same event reported multiple times)
silver_events = silver_events.drop_duplicates(subset=['globaleventid'], keep='first')

logger.info(f"Silver events shape: {silver_events.shape}")
print(f"Date range: {silver_events['date'].min()} to {silver_events['date'].max()}")
print(f"Total events: {len(silver_events)}")


2025-12-20 14:35:05,385 - INFO - Processing silver_events...
2025-12-20 14:35:05,393 - INFO - Dropped 0 rows with missing titles
2025-12-20 14:35:05,413 - INFO - Silver events shape: (1000, 25)


Date range: 2015-12-13 00:00:00 to 2025-12-13 00:00:00
Total events: 1000


In [81]:
# Process silver_gkg
logger.info("Processing silver_gkg...")

# Drop rows where title is NaN
silver_gkg = bronze_gkg.dropna(subset=['title']).copy()
logger.info(f"Dropped {len(bronze_gkg) - len(silver_gkg)} rows with missing titles")

# Select relevant columns for financial prediction
gkg_columns = [
    'gkgrecordid',
    'date',
    'sourcecommonname',
    'documentidentifier',
    'themes',
    'v2themes',
    'locations',
    'v2locations',
    'persons',
    'v2persons',
    'organizations',
    'v2organizations',
    'v2tone',
    'allnames',
    'amounts',
    'title'
]

silver_gkg = silver_gkg[gkg_columns]

# Convert date to datetime (GDELT GKG date format: YYYYMMDDHHMMSS)
silver_gkg['datetime'] = pd.to_datetime(silver_gkg['date'], format='%Y%m%d%H%M%S', errors='coerce')
silver_gkg['date_only'] = silver_gkg['datetime'].dt.date

# Drop rows with invalid dates
invalid_dates = silver_gkg['datetime'].isna().sum()
if invalid_dates > 0:
    logger.warning(f"Found {invalid_dates} invalid dates, dropping them")
    silver_gkg = silver_gkg.dropna(subset=['datetime'])

# Sort by datetime
silver_gkg = silver_gkg.sort_values('datetime').reset_index(drop=True)

# Remove duplicate records
silver_gkg = silver_gkg.drop_duplicates(subset=['gkgrecordid'], keep='first')

# Parse tone (format: "tone,positive,negative,polarity,activity_reference_density,self_reference_density,word_count")
if 'v2tone' in silver_gkg.columns:
    tone_split = silver_gkg['v2tone'].str.split(',', expand=True)
    if tone_split.shape[1] >= 7:
        silver_gkg['tone'] = pd.to_numeric(tone_split[0], errors='coerce')
        silver_gkg['tone_positive'] = pd.to_numeric(tone_split[1], errors='coerce')
        silver_gkg['tone_negative'] = pd.to_numeric(tone_split[2], errors='coerce')
        silver_gkg['tone_polarity'] = pd.to_numeric(tone_split[3], errors='coerce')

logger.info(f"Silver GKG shape: {silver_gkg.shape}")
print(f"Date range: {silver_gkg['datetime'].min()} to {silver_gkg['datetime'].max()}")
print(f"Total GKG records: {len(silver_gkg)}")


2025-12-20 14:35:05,430 - INFO - Processing silver_gkg...
2025-12-20 14:35:05,436 - INFO - Dropped 0 rows with missing titles
2025-12-20 14:35:05,458 - INFO - Silver GKG shape: (1000, 22)


Date range: 2025-12-04 00:00:00 to 2025-12-13 00:00:00
Total GKG records: 1000


In [83]:
silver_gkg[['themes', 'organizations', 'v2organizations']]

,themes,organizations,v2organizations
0,EPU_POLICY;EPU_POLICY_FEDERAL_RESERVE;EPU_CATS...,american express;broadcom;marvell technology;m...,"American Express,759;Broadcom,935;Marvell Tech..."
1,WB_1331_HEALTH_TECHNOLOGIES;WB_2453_ORGANIZED_...,walmart;drug administration;great lakes cheese...,"Walmart,452;Drug Administration,177;Great Lake..."
2,TAX_FNCACT;TAX_FNCACT_ACTOR;TAX_FNCACT_COWBOY;...,netflix,"Netflix,655;Netflix,1064"
3,MARITIME;CLOSURE;NEW_CONSTRUCTION;UNGP_FORESTS...,NaN,NaN
4,WB_1331_HEALTH_TECHNOLOGIES;WB_2453_ORGANIZED_...,united states;publix;great lakes cheese co;wal...,"United States,423;Publix,470;Great Lakes Chees..."
...,...,...,...
995,GENERAL_GOVERNMENT;EPU_ECONOMY;EPU_ECONOMY_HIS...,united nations;u s senate;united states;visa;s...,"United Nations,4501;United States,2691;Visa,10..."
996,MEDIA_MSM;UNGP_FORESTS_RIVERS_OCEANS;CRISISLEX...,NaN,NaN
997,TAX_FNCACT;TAX_FNCACT_LEFTIST;SOC_FASCISM;TAX_...,white house;disney;bloomberg,"White House,518;White House,2283;Disney,69;Dis..."
998,GENERAL_GOVERNMENT;EPU_ECONOMY;EPU_ECONOMY_HIS...,syria emergency task;visa;united states;u s se...,"Syria Emergency Task,3679;Visa,1009;United Sta..."


In [ ]:
# Process silver_stooq
logger.info("Processing silver_stooq...")

silver_stooq = bronze_stooq.copy()

# Convert timestamp to datetime
silver_stooq['date'] = pd.to_datetime(silver_stooq['timestamp'], errors='coerce')

# Drop rows with invalid dates
invalid_dates = silver_stooq['date'].isna().sum()
if invalid_dates > 0:
    logger.warning(f"Found {invalid_dates} invalid dates, dropping them")
    silver_stooq = silver_stooq.dropna(subset=['date'])

# Sort by date and symbol
silver_stooq = silver_stooq.sort_values(['symbol', 'date']).reset_index(drop=True)

# Remove duplicates (same symbol and date)
silver_stooq = silver_stooq.drop_duplicates(subset=['symbol', 'date'], keep='last')

# Add calculated features
silver_stooq['daily_return'] = silver_stooq.groupby('symbol')['close'].pct_change()
silver_stooq['price_range'] = silver_stooq['high'] - silver_stooq['low']
silver_stooq['price_change'] = silver_stooq['close'] - silver_stooq['open']

# Select relevant columns
stooq_columns = [
    'date',
    'symbol',
    'open',
    'high',
    'low',
    'close',
    'volume',
    'daily_return',
    'price_range',
    'price_change'
]

silver_stooq = silver_stooq[stooq_columns]

logger.info(f"Silver stooq shape: {silver_stooq.shape}")
print(f"Date range: {silver_stooq['date'].min()} to {silver_stooq['date'].max()}")
print(f"Total stock records: {len(silver_stooq)}")
print(f"Unique symbols: {silver_stooq['symbol'].nunique()}")


2025-12-20 12:58:03,219 - INFO - Processing silver_stooq...
2025-12-20 12:58:03,229 - INFO - Silver stooq shape: (126, 10)


Date range: 2025-11-13 00:00:00 to 2025-12-12 00:00:00
Total stock records: 126
Unique symbols: 6


In [ ]:
# Display sample data
print("\n=== Silver Events Sample ===")
print(silver_events[['date', 'title', 'goldsteinscale', 'avgtone', 'nummentions']].head())

print("\n=== Silver GKG Sample ===")
print(silver_gkg[['datetime', 'title', 'tone', 'sourcecommonname']].head())

print("\n=== Silver Stooq Sample ===")
print(silver_stooq[['date', 'symbol', 'close', 'daily_return', 'volume']].head())



=== Silver Events Sample ===
        date title  goldsteinscale   avgtone  nummentions
0 2015-12-13                   7.4 -2.368866           10
1 2024-12-04                 -10.0 -4.528986            1
2 2024-12-04                 -10.0 -4.528986            1
3 2024-12-04                 -10.0 -4.528986            3
4 2024-12-05                   5.0 -2.216749           20

=== Silver GKG Sample ===
    datetime title      tone  sourcecommonname
0 2025-12-04       -1.604278  athens-times.com
1 2025-12-04        0.628931        iheart.com
2 2025-12-04        0.715137  yakimaherald.com
3 2025-12-04       -0.252525          wtae.com
4 2025-12-04        0.628931        iheart.com

=== Silver Stooq Sample ===
        date symbol   close  daily_return    volume
0 2025-11-13   AAPL  272.95           NaN  49602794
1 2025-11-14   AAPL  272.41     -0.001978  47431331
2 2025-11-17   AAPL  267.46     -0.018171  45018260
3 2025-11-18   AAPL  267.44     -0.000075  45677278
4 2025-11-19   AAPL  268

In [ ]:
# Check data quality
print("\n=== Data Quality Summary ===")
print(f"\nSilver Events:")
print(f"  - Missing values: {silver_events.isnull().sum().sum()}")
print(f"  - Goldstein scale range: [{silver_events['goldsteinscale'].min():.2f}, {silver_events['goldsteinscale'].max():.2f}]")
print(f"  - Average tone range: [{silver_events['avgtone'].min():.2f}, {silver_events['avgtone'].max():.2f}]")

print(f"\nSilver GKG:")
print(f"  - Missing values: {silver_gkg.isnull().sum().sum()}")
if 'tone' in silver_gkg.columns:
    print(f"  - Tone range: [{silver_gkg['tone'].min():.2f}, {silver_gkg['tone'].max():.2f}]")

print(f"\nSilver Stooq:")
print(f"  - Missing values: {silver_stooq.isnull().sum().sum()}")
print(f"  - Price range: ${silver_stooq['close'].min():.2f} - ${silver_stooq['close'].max():.2f}")



=== Data Quality Summary ===

Silver Events:
  - Missing values: 2266
  - Goldstein scale range: [-10.00, 10.00]
  - Average tone range: [-15.85, 11.45]

Silver GKG:
  - Missing values: 2111
  - Tone range: [-16.15, 22.63]

Silver Stooq:
  - Missing values: 6
  - Price range: $217.14 - $673.42


# 3. Gold data


In [ ]:
# 1. Process events with lightweight features
gold_events = silver_events.copy()

# Simple vectorized text features (no .apply())
gold_events['title_length'] = gold_events['title'].str.len()
gold_events['title_word_count'] = gold_events['title'].str.split().str.len()

# Use GDELT's built-in sentiment (already computed!)
gold_events_daily = gold_events.groupby('date').agg({
    'goldsteinscale': ['mean', 'std', 'min', 'max'],
    'avgtone': ['mean', 'std', 'min', 'max'],  # GDELT's sentiment
    'nummentions': 'sum',
    'numsources': 'sum',
    'numarticles': 'sum',
    'title_length': 'mean',
    'title_word_count': 'mean',
    'globaleventid': 'count'
}).reset_index()

# Flatten column names
gold_events_daily.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                              for col in gold_events_daily.columns.values]
gold_events_daily.rename(columns={'globaleventid_count': 'event_count'}, inplace=True)
gold_events_daily = gold_events_daily.fillna(0)

print(f"Gold events daily shape: {gold_events_daily.shape}")
print(f"Date range: {gold_events_daily['date'].min()} to {gold_events_daily['date'].max()}")
print(f"Total days: {len(gold_events_daily)}")

Gold events daily shape: (35, 15)
Date range: 2015-12-13 00:00:00 to 2025-12-13 00:00:00
Total days: 35


In [ ]:
# 2. Process GKG with lightweight features
gold_gkg = silver_gkg.copy()

# Simple vectorized text features
gold_gkg['title_length'] = gold_gkg['title'].str.len()
gold_gkg['title_word_count'] = gold_gkg['title'].str.split().str.len()
gold_gkg['date_for_agg'] = gold_gkg['datetime'].dt.date

# Use GDELT's built-in tone metrics
gold_gkg_daily = gold_gkg.groupby('date_for_agg').agg({
    'tone': ['mean', 'std', 'min', 'max'],  # GDELT's sentiment
    'tone_positive': ['mean', 'std'],
    'tone_negative': ['mean', 'std'],
    'tone_polarity': ['mean', 'std'],
    'title_length': 'mean',
    'title_word_count': 'mean',
    'gkgrecordid': 'count'
}).reset_index()

# Flatten column names
gold_gkg_daily.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                           for col in gold_gkg_daily.columns.values]
gold_gkg_daily.rename(columns={
    'date_for_agg': 'date',
    'gkgrecordid_count': 'gkg_count'
}, inplace=True)
gold_gkg_daily['date'] = pd.to_datetime(gold_gkg_daily['date'])
gold_gkg_daily = gold_gkg_daily.fillna(0)

logger.info(f"Gold GKG daily shape: {gold_gkg_daily.shape}")
print(f"Date range: {gold_gkg_daily['date'].min()} to {gold_gkg_daily['date'].max()}")
print(f"Total days: {len(gold_gkg_daily)}")

2025-12-20 13:06:51,148 - INFO - Gold GKG daily shape: (10, 14)


Date range: 2025-12-04 00:00:00 to 2025-12-13 00:00:00
Total days: 10


In [ ]:
gold_gkg.columns = date, theme, org, title

gold_stooq = 

merge 2 table above

gold_symbol_datasets['AAPL']

In [ ]:
# 3. Gold stooq (copy from silver)
gold_stooq = silver_stooq.copy()

print(f"Gold stooq shape: {gold_stooq.shape}")
print(f"Unique symbols: {gold_stooq['symbol'].unique()}")

# 4. Create gold_symbol datasets
symbols = gold_stooq['symbol'].unique()
gold_symbol_datasets = {}

for symbol in symbols:
    print(f"Processing gold_{symbol.lower()}...")
    
    symbol_stock = gold_stooq[gold_stooq['symbol'] == symbol].copy()
    
    # Merge with events and GKG
    gold_symbol = pd.merge(symbol_stock, gold_events_daily, on='date', how='left')
    gold_symbol = pd.merge(gold_symbol, gold_gkg_daily, on='date', how='left')
    
    # Sort and fill
    gold_symbol = gold_symbol.sort_values('date').reset_index(drop=True)
    cols_to_fill = [col for col in gold_symbol.columns if col not in ['date', 'symbol']]
    gold_symbol[cols_to_fill] = gold_symbol[cols_to_fill].fillna(method='ffill').fillna(0)
    
    # Add time features (vectorized)
    gold_symbol['day_of_week'] = gold_symbol['date'].dt.dayofweek
    gold_symbol['month'] = gold_symbol['date'].dt.month
    gold_symbol['quarter'] = gold_symbol['date'].dt.quarter
    
    # Lagged features (vectorized)
    gold_symbol['prev_close'] = gold_symbol['close'].shift(1)
    gold_symbol['prev_volume'] = gold_symbol['volume'].shift(1)
    gold_symbol['prev_daily_return'] = gold_symbol['daily_return'].shift(1)
    
    # Rolling features (vectorized)
    gold_symbol['close_ma7'] = gold_symbol['close'].rolling(window=7, min_periods=1).mean()
    gold_symbol['volume_ma7'] = gold_symbol['volume'].rolling(window=7, min_periods=1).mean()
    
    gold_symbol_datasets[symbol] = gold_symbol
    print(f"  - Shape: {gold_symbol.shape}")

print(f"\n✓ Gold datasets created (Spark-optimized)!")
print(f"  - No external dependencies")
print(f"  - Uses GDELT's built-in sentiment")
print(f"  - Vectorized operations only")
print(f"  - Total symbols: {len(gold_symbol_datasets)}")

Gold stooq shape: (126, 10)
Unique symbols: ['AAPL' 'AMZN' 'GOOGL' 'META' 'MSFT' 'TSLA']
Processing gold_aapl...
  - Shape: (21, 45)
Processing gold_amzn...
  - Shape: (21, 45)
Processing gold_googl...
  - Shape: (21, 45)
Processing gold_meta...
  - Shape: (21, 45)
Processing gold_msft...
  - Shape: (21, 45)
Processing gold_tsla...
  - Shape: (21, 45)

✓ Gold datasets created (Spark-optimized)!
  - No external dependencies
  - Uses GDELT's built-in sentiment
  - Vectorized operations only
  - Total symbols: 6


/tmp/ipykernel_19737/4230694238.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gold_symbol[cols_to_fill] = gold_symbol[cols_to_fill].fillna(method='ffill').fillna(0)
/tmp/ipykernel_19737/4230694238.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gold_symbol[cols_to_fill] = gold_symbol[cols_to_fill].fillna(method='ffill').fillna(0)
/tmp/ipykernel_19737/4230694238.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gold_symbol[cols_to_fill] = gold_symbol[cols_to_fill].fillna(method='ffill').fillna(0)
/tmp/ipykernel_19737/4230694238.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gold_symbol[cols_to_fill] = gol

In [ ]:
gold_symbol_datasets['AAPL'].columns

Index(['date', 'symbol', 'open', 'high', 'low', 'close', 'volume',
       'daily_return', 'price_range', 'price_change', 'goldsteinscale_mean',
       'goldsteinscale_std', 'goldsteinscale_min', 'goldsteinscale_max',
       'avgtone_mean', 'avgtone_std', 'avgtone_min', 'avgtone_max',
       'nummentions_sum', 'numsources_sum', 'numarticles_sum',
       'title_length_mean_x', 'title_word_count_mean_x', 'event_count',
       'tone_mean', 'tone_std', 'tone_min', 'tone_max', 'tone_positive_mean',
       'tone_positive_std', 'tone_negative_mean', 'tone_negative_std',
       'tone_polarity_mean', 'tone_polarity_std', 'title_length_mean_y',
       'title_word_count_mean_y', 'gkg_count', 'day_of_week', 'month',
       'quarter', 'prev_close', 'prev_volume', 'prev_daily_return',
       'close_ma7', 'volume_ma7'],
      dtype='object')

In [ ]:
gold_symbol_datasets['AAPL'].head()

,date,symbol,open,high,low,close,volume,daily_return,price_range,price_change,...,title_word_count_mean_y,gkg_count,day_of_week,month,quarter,prev_close,prev_volume,prev_daily_return,close_ma7,volume_ma7
0,2025-11-13,AAPL,274.110,276.699,272.09,272.95,49602794,0.000000,4.609,-1.160,...,0.0,0.0,3,11,4,NaN,NaN,NaN,272.950,49602794.00
1,2025-11-14,AAPL,271.050,275.960,269.60,272.41,47431331,-0.001978,6.360,1.360,...,0.0,0.0,4,11,4,272.95,49602794.0,0.000000,272.680,48517062.50
2,2025-11-17,AAPL,268.815,270.490,265.73,267.46,45018260,-0.018171,4.760,-1.355,...,0.0,0.0,0,11,4,272.41,47431331.0,-0.001978,270.940,47350795.00
3,2025-11-18,AAPL,269.990,270.710,265.32,267.44,45677278,-0.000075,5.390,-2.550,...,0.0,0.0,1,11,4,267.46,45018260.0,-0.018171,270.065,46932415.75
4,2025-11-19,AAPL,265.525,272.210,265.50,268.56,40424492,0.004188,6.710,3.035,...,0.0,0.0,2,11,4,267.44,45677278.0,-0.000075,269.764,45630831.00


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/mnt/c/Coding/bdsp-test/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/mnt/c/Coding/bdsp-test/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/mnt/c/Coding/bdsp-test/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Coding/bdsp-test/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/mnt/c/Coding/bdsp-test/.venv/lib/python3.12/site-packages/zmq/event

In [ ]:
# Display sample gold data
print("\n=== Gold Events Daily Sample ===")
display_cols = ['date', 'goldsteinscale_mean', 'avgtone_mean', 'event_count']
print(gold_events_daily[display_cols].head())

print("\n=== Gold GKG Daily Sample ===")
display_cols = ['date', 'tone_mean', 'gkg_count', 'title_length_mean']
print(gold_gkg_daily[display_cols].head())

# Display one example gold_symbol dataset
if len(gold_symbol_datasets) > 0:
    example_symbol = list(gold_symbol_datasets.keys())[0]
    example_df = gold_symbol_datasets[example_symbol]
    print(f"\n=== Gold {example_symbol} Sample (first 5 rows) ===")
    display_cols = ['date', 'symbol', 'close', 'daily_return', 'volume',
                    'goldsteinscale_mean', 'avgtone_mean', 'tone_mean']
    available_cols = [col for col in display_cols if col in example_df.columns]
    print(example_df[available_cols].head())


=== Gold Events Daily Sample ===
        date  goldsteinscale_mean  avgtone_mean  event_count
0 2015-12-13                  7.4     -2.368866            1
1 2024-12-04                -10.0     -4.528986            3
2 2024-12-05                  5.0     -2.216749            1
3 2024-12-06                  1.9     -0.329556            6
4 2024-12-07                 -7.6      1.160338            4

=== Gold GKG Daily Sample ===
        date  tone_mean  gkg_count  title_length_mean
0 2025-12-04  -0.333418        100                0.0
1 2025-12-05  -0.179916        100                0.0
2 2025-12-06   0.026423        100                0.0
3 2025-12-07  -1.320313        100                0.0
4 2025-12-08  -0.352495        100                0.0

=== Gold AAPL Sample (first 5 rows) ===
        date symbol   close  daily_return    volume  goldsteinscale_mean  \
0 2025-11-13   AAPL  272.95      0.000000  49602794            -3.857143   
1 2025-11-14   AAPL  272.41     -0.001978  47431331 

In [ ]:
gold_symbol_datasets['AAPL']

,date,symbol,open,high,low,close,volume,daily_return,price_range,price_change,...,title_word_count_mean_y,gkg_count,day_of_week,month,quarter,prev_close,prev_volume,prev_daily_return,close_ma7,volume_ma7
0,2025-11-13,AAPL,274.110,276.6990,272.0900,272.95,49602794,0.000000,4.6090,-1.160,...,0.0,0.0,3,11,4,NaN,NaN,NaN,272.950000,4.960279e+07
1,2025-11-14,AAPL,271.050,275.9600,269.6000,272.41,47431331,-0.001978,6.3600,1.360,...,0.0,0.0,4,11,4,272.95,49602794.0,0.000000,272.680000,4.851706e+07
2,2025-11-17,AAPL,268.815,270.4900,265.7300,267.46,45018260,-0.018171,4.7600,-1.355,...,0.0,0.0,0,11,4,272.41,47431331.0,-0.001978,270.940000,4.735080e+07
3,2025-11-18,AAPL,269.990,270.7100,265.3200,267.44,45677278,-0.000075,5.3900,-2.550,...,0.0,0.0,1,11,4,267.46,45018260.0,-0.018171,270.065000,4.693242e+07
4,2025-11-19,AAPL,265.525,272.2100,265.5000,268.56,40424492,0.004188,6.7100,3.035,...,0.0,0.0,2,11,4,267.44,45677278.0,-0.000075,269.764000,4.563083e+07
5,2025-11-20,AAPL,270.830,275.4300,265.9200,266.25,45823568,-0.008601,9.5100,-4.580,...,0.0,0.0,3,11,4,268.56,40424492.0,0.004188,269.178333,4.566295e+07
6,2025-11-21,AAPL,265.950,273.3300,265.6700,271.49,59030832,0.019681,7.6600,5.540,...,0.0,0.0,4,11,4,266.25,45823568.0,-0.008601,269.508571,4.757265e+07
7,2025-11-24,AAPL,270.900,277.0000,270.9000,275.92,65585796,0.016317,6.1000,5.020,...,0.0,0.0,0,11,4,271.49,59030832.0,0.019681,269.932857,4.985594e+07
8,2025-11-25,AAPL,275.270,280.3800,275.2500,276.97,46914220,0.003805,5.1300,1.700,...,0.0,0.0,1,11,4,275.92,65585796.0,0.016317,270.584286,4.978206e+07
9,2025-11-26,AAPL,276.960,279.5300,276.6300,277.55,33431423,0.002094,2.9000,0.590,...,0.0,0.0,2,11,4,276.97,46914220.0,0.003805,272.025714,4.812680e+07


In [ ]:
# Check data quality for gold datasets
print("\n=== Gold Data Quality Summary ===")

print(f"\nGold Events Daily:")
print(f"  - Shape: {gold_events_daily.shape}")
print(f"  - Missing values: {gold_events_daily.isnull().sum().sum()}")
print(f"  - Date coverage: {len(gold_events_daily)} days")

print(f"\nGold GKG Daily:")
print(f"  - Shape: {gold_gkg_daily.shape}")
print(f"  - Missing values: {gold_gkg_daily.isnull().sum().sum()}")
print(f"  - Date coverage: {len(gold_gkg_daily)} days")

print(f"\nGold Stooq:")
print(f"  - Shape: {gold_stooq.shape}")
print(f"  - Missing values: {gold_stooq.isnull().sum().sum()}")
print(f"  - Symbols: {gold_stooq['symbol'].nunique()}")

print(f"\nGold Symbol Datasets:")
for symbol, df in gold_symbol_datasets.items():
    missing = df.isnull().sum().sum()
    print(f"  - gold_{symbol.lower()}: {df.shape}, missing values: {missing}")
    
print("\n✓ All gold datasets are ready for train/val/test split!")


=== Gold Data Quality Summary ===

Gold Events Daily:
  - Shape: (35, 15)
  - Missing values: 0
  - Date coverage: 35 days

Gold GKG Daily:
  - Shape: (10, 14)
  - Missing values: 0
  - Date coverage: 10 days

Gold Stooq:
  - Shape: (126, 10)
  - Missing values: 6
  - Symbols: 6

Gold Symbol Datasets:
  - gold_aapl: (21, 45), missing values: 3
  - gold_amzn: (21, 45), missing values: 3
  - gold_googl: (21, 45), missing values: 3
  - gold_meta: (21, 45), missing values: 3
  - gold_msft: (21, 45), missing values: 3
  - gold_tsla: (21, 45), missing values: 3

✓ All gold datasets are ready for train/val/test split!


In [ ]:
# Export gold datasets to CSV for later use
import os

output_dir = '/mnt/c/Coding/bdsp-test/data-csv/gold'
os.makedirs(output_dir, exist_ok=True)

# Save aggregated datasets
gold_events_daily.to_csv(f'{output_dir}/gold_events_daily.csv', index=False)
gold_gkg_daily.to_csv(f'{output_dir}/gold_gkg_daily.csv', index=False)
gold_stooq.to_csv(f'{output_dir}/gold_stooq.csv', index=False)

# Save individual symbol datasets
for symbol, df in gold_symbol_datasets.items():
    filename = f'{output_dir}/gold_{symbol.lower()}.csv'
    df.to_csv(filename, index=False)
    logger.info(f"Saved {filename}")

print(f"\n✓ All gold datasets saved to {output_dir}")
print(f"  - gold_events_daily.csv")
print(f"  - gold_gkg_daily.csv")
print(f"  - gold_stooq.csv")
for symbol in gold_symbol_datasets.keys():
    print(f"  - gold_{symbol.lower()}.csv")

2025-12-20 12:58:03,482 - INFO - Saved /mnt/c/Coding/bdsp-test/data-csv/gold/gold_aapl.csv
2025-12-20 12:58:03,489 - INFO - Saved /mnt/c/Coding/bdsp-test/data-csv/gold/gold_amzn.csv
2025-12-20 12:58:03,496 - INFO - Saved /mnt/c/Coding/bdsp-test/data-csv/gold/gold_googl.csv
2025-12-20 12:58:03,503 - INFO - Saved /mnt/c/Coding/bdsp-test/data-csv/gold/gold_meta.csv
2025-12-20 12:58:03,510 - INFO - Saved /mnt/c/Coding/bdsp-test/data-csv/gold/gold_msft.csv
2025-12-20 12:58:03,516 - INFO - Saved /mnt/c/Coding/bdsp-test/data-csv/gold/gold_tsla.csv



✓ All gold datasets saved to /mnt/c/Coding/bdsp-test/data-csv/gold
  - gold_events_daily.csv
  - gold_gkg_daily.csv
  - gold_stooq.csv
  - gold_aapl.csv
  - gold_amzn.csv
  - gold_googl.csv
  - gold_meta.csv
  - gold_msft.csv
  - gold_tsla.csv
